# Day 1 · Fundamentals and reading the Spark UI

**Objective:** predict how Spark will execute a query *before* running it, then confirm it in the UI.

Reference: `knowledge_hub/skills/spark/01_fundamentals_and_spark_ui.md`

---

### Before you start

The Spark UI serves on `localhost:4040`, and this container is not your browser's host.
Forward port **4040** (VS Code **PORTS** panel) before running anything below, or you will
be doing a UI-reading exercise with no UI.

### The one rule to hold onto today

> **number of stages = number of shuffles + 1**

Everything in the Stages tab makes sense once that is in your head.

## Setup

In [1]:
# Plumbing, not Spark. The notebook's working directory is notebooks/, so the
# repo root has to be importable before `from src...` resolves.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
# Open the session.
#   - import get_spark and YELLOW from src.session
#   - open a session named "day1"
#   - get_spark prints the UI URL; open that tab and leave it open all week

from src.session import get_spark, YELLOW

spark = get_spark("day1")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 05:14:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.9 | AQE=False | UI http://wc-allauddin-shaik-shared-vpc:4040


---
## E1 · Confirm the schema

⚠️ **Do this before anything else.** Two things come out of it:

1. **The Day 5 branch.** If the files carry `pickup_latitude` / `pickup_longitude`, H3 indexing
   is a direct call. If they carry only `PULocationID`, you have to polyfill the taxi zone
   polygons into hexagons instead, which is more work and a better story.
2. **The baseline.** Every benchmark later says "faster than what". Row count, partition count
   and size on disk are that "what".

In [ ]:
# Read every monthly parquet under YELLOW into a DataFrame called df.
#
# Then look at the Jobs tab. You will probably see a small job appear even though
# you have not called an action yet. Work out why before reading on: reading Parquet
# is not fully lazy, because Spark has to open the file footers to learn the schema.

In [ ]:
# Print the schema.
#
# Scan for: pickup_latitude / pickup_longitude   vs   PULocationID / DOLocationID

In [3]:
df = spark.read.parquet(YELLOW)

In [4]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 N|         142|         239|           1|        8.6|  3.5|    0.5|       2.

In [ ]:
df.printSchema()

In [5]:
%%time
df.count()

CPU times: user 2.4 ms, sys: 1.87 ms, total: 4.27 ms
Wall time: 1.62 s


9554778

In [6]:
df.rdd.getNumPartitions()

3

**Fill this in:**

- Location columns present: `PULocationID`, `DOLocationID`. **No lat/lon.**
- Therefore Day 5 is: zone shapefile polyfill into H3 cells, many-to-many, apportionment to decide.


In [ ]:
# Count the rows. This is the first real job of the week.
#
# While it runs, open the Jobs tab, click into the job, then into the stage.
# Note the task count. That is your partition count, and it is the number every
# Day 2 experiment moves.

In [ ]:
# Confirm the partition count directly, and check it matches the task count you
# just read off the UI. If it does, you are reading the UI correctly.

**Fill this in:**

| | |
|---|---|
| Rows (3 months) | 9554778|
| Partitions | 3|
| Tasks in the count job |4 |
| Wall time | 2.53 sec|
| Size on disk | 153 MB |

---
## E2 · Lazy versus eager

The point: a transformation builds a plan and runs nothing. Only an action executes.

The practical consequence, which is what makes it worth knowing: **the slow line of code is
usually not the slow operation.** The action at the end gets blamed for the ten transformations
above it.

In [7]:
# Build a filtered DataFrame: trips with trip_distance > 5. Do not call an action.
#
# Check the Jobs tab. Confirm no new job appeared.
df.filter("trip_distance > 5")

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double]

In [8]:
# Now call an action on it. Watch a job appear.
df.filter("trip_distance > 5").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-03-01 00:50:42|  2024-03-01 01:10:40|              1|         5.04|         1|                 N|         238|         159|           2|       25.4|  1.0|    0.5|       0.

---
## E3 · Predict, then verify

⭐ This is the exercise that matters today. **Write your predictions in the table below before
running a single cell.** Being wrong and working out why is the entire point; running first and
rationalising afterwards teaches you nothing.

| Query | Predicted stages | Actual | Why |
|---|---|---|---|
| A · filter, select, count | 2|2 | each partition computes a partial answer independently, then those partials have to meet in one place to be merged. |
| B · groupBy PULocationID, count, collect | 2| 2| same |
| C · groupBy, count, orderBy desc, take 10 | 2| 2| same |

In [9]:
# Query A: filter on trip_distance > 5, select trip_distance, count.
df.filter("trip_distance > 5").select("trip_distance").count()

1496600

In [10]:
# Query B: group by PULocationID, count, collect.
df.groupBy("PULocationID").count().collect()

[Row(PULocationID=148, count=105120),
 Row(PULocationID=243, count=2180),
 Row(PULocationID=31, count=50),
 Row(PULocationID=137, count=107172),
 Row(PULocationID=85, count=870),
 Row(PULocationID=251, count=10),
 Row(PULocationID=65, count=5739),
 Row(PULocationID=255, count=4670),
 Row(PULocationID=53, count=191),
 Row(PULocationID=133, count=515),
 Row(PULocationID=78, count=755),
 Row(PULocationID=108, count=402),
 Row(PULocationID=155, count=783),
 Row(PULocationID=211, count=73173),
 Row(PULocationID=193, count=6108),
 Row(PULocationID=34, count=278),
 Row(PULocationID=126, count=451),
 Row(PULocationID=101, count=174),
 Row(PULocationID=115, count=16),
 Row(PULocationID=81, count=431),
 Row(PULocationID=183, count=197),
 Row(PULocationID=28, count=1300),
 Row(PULocationID=210, count=736),
 Row(PULocationID=76, count=3284),
 Row(PULocationID=26, count=748),
 Row(PULocationID=27, count=16),
 Row(PULocationID=192, count=213),
 Row(PULocationID=159, count=1114),
 Row(PULocationID=44

In [11]:
# Query C: group by PULocationID, count, order by count descending, take 10.
df.groupBy("PULocationID").count().orderBy("count", ascending=False).take(10)

[Row(PULocationID=161, count=453826),
 Row(PULocationID=237, count=439139),
 Row(PULocationID=132, count=429746),
 Row(PULocationID=236, count=416509),
 Row(PULocationID=162, count=336460),
 Row(PULocationID=230, count=331570),
 Row(PULocationID=186, count=319706),
 Row(PULocationID=142, count=316314),
 Row(PULocationID=138, count=284362),
 Row(PULocationID=239, count=281730)]

---
## E4 · Read the plans

`explain("formatted")`, read bottom-up. What to look for:

- `FileScan parquet` and whether `PushedFilters` shows your filter reached the scan
- **`Exchange` means shuffle.** Count them.
- `BroadcastHashJoin` versus `SortMergeJoin`
- `ReadSchema`, and whether it is reading only the columns you need

Then check the rule: `Exchange` count + 1 should equal the stage count you saw in the UI.

In [ ]:
# Print the formatted plan for each of A, B and C. Count the Exchange lines in each.
df.explain("formatted")

In [12]:
df.filter("trip_distance > 5").select("trip_distance").groupBy().count().explain("formatted")

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [trip_distance#4]
Batched: true
Location: InMemoryFileIndex [file:/home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-01.parquet, ... 2 entries]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,5.0)]
ReadSchema: struct<trip_distance:double>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [trip_distance#4]

(3) Filter [codegen id : 1]
Input [1]: [trip_distance#4]
Condition : (isnotnull(trip_distance#4) AND (trip_distance#4 > 5.0))

(4) Project [codegen id : 1]
Output: []
Input [1]: [trip_distance#4]

(5) HashAggregate [codegen id : 1]
Input: []
Keys: []
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#279L]
Results [1]: [count#280L]

(6) Exchange
Input [1]: [count#280L]
Arguments: SinglePartition, ENSURE_REQ

In [14]:
df.groupBy("PULocationID").count().explain("formatted")

== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-01.parquet, ... 2 entries]
ReadSchema: struct<PULocationID:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [PULocationID#7]

(3) HashAggregate [codegen id : 1]
Input [1]: [PULocationID#7]
Keys [1]: [PULocationID#7]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#329L]
Results [2]: [PULocationID#7, count#330L]

(4) Exchange
Input [2]: [PULocationID#7, count#330L]
Arguments: hashpartitioning(PULocationID#7, 200), ENSURE_REQUIREMENTS, [plan_id=279]

(5) HashAggregate [codegen id : 2]
Input [2]: [PULocationID#7, count#330L]
Keys [1]: [PULocationID#7]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#325L]
Results [2]: [PULocationID#7, count(1)#325L AS coun

In [15]:
df.groupBy("PULocationID").count().orderBy("count", ascending=False).explain("formatted")

== Physical Plan ==
* Sort (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- Exchange (4)
         +- * HashAggregate (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-01.parquet, ... 2 entries]
ReadSchema: struct<PULocationID:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [PULocationID#7]

(3) HashAggregate [codegen id : 1]
Input [1]: [PULocationID#7]
Keys [1]: [PULocationID#7]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#354L]
Results [2]: [PULocationID#7, count#355L]

(4) Exchange
Input [2]: [PULocationID#7, count#355L]
Arguments: hashpartitioning(PULocationID#7, 200), ENSURE_REQUIREMENTS, [plan_id=320]

(5) HashAggregate [codegen id : 2]
Input [2]: [PULocationID#7, count#355L]
Keys [1]: [PULocationID#7]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#

---
## E5 · Predicate and column pushdown

This is what "columnar format" actually buys you, and it is measurable rather than theoretical.

In [17]:
# Select only PULocationID, filter on trip_distance > 5, and print the formatted plan.
#
# Find PushedFilters and ReadSchema. Confirm two things:
#   - Spark pushed the filter down into the Parquet scan rather than reading everything
#     and filtering afterwards
#   - it is reading fewer columns than the file contains

df.filter("trip_distance > 5").select('PULocationID').explain('formatted')

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [trip_distance#4, PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-01.parquet, ... 2 entries]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,5.0)]
ReadSchema: struct<trip_distance:double,PULocationID:int>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [trip_distance#4, PULocationID#7]

(3) Filter [codegen id : 1]
Input [2]: [trip_distance#4, PULocationID#7]
Condition : (isnotnull(trip_distance#4) AND (trip_distance#4 > 5.0))

(4) Project [codegen id : 1]
Output [1]: [PULocationID#7]
Input [2]: [trip_distance#4, PULocationID#7]




### Can I say these out loud, unaided?

Answer each one out loud **before** reading the block under it. The wording of the answers is
deliberate: rule first, then a measured number from this notebook. "Each partition counted its rows,
then 177 bytes crossed the shuffle to be summed" is a different class of answer from "count shuffles".

---

**1. Why does this job have N stages?**  `[ ]`

> Spark cuts the plan into stages at every shuffle, so **stages = shuffles + 1**. A shuffle happens
> when producing the answer requires combining data across partitions. My `count()` had two stages:
> each of the 3 partitions counted its own rows, then those 3 partial counts had to meet in one
> place to be summed. 177 bytes crossed that boundary.

---

**2. What is a partition, and how many does this job have?**  `[ ]`

> A partition is **a chunk of the data**. Not a machine, not a worker. It is the unit of
> parallelism: **one task processes one partition on one core.**
>
> I had 3 partitions, so at most 3 tasks could run at once, and with 2 cores that meant 2 at a time
> and one waiting. Stage 4 later ran **200 tasks on 2 cores**, which is only coherent if partitions
> are data and cores are workers.
>
> Size is not simply 128 MB. It is
> `min(maxPartitionBytes 128MB, max(openCostInBytes 4MB, totalBytes / cores))`.
> For my 165 MiB across 2 cores that came to **82.5 MiB**, so each ~50 MB file stayed whole and I
> got 3. On a 10-core machine the same files would give 9 or 10.

---

**3. Name five operations that shuffle and five that do not**  `[ ]`

> **Shuffle (wide):** `groupBy().agg()`, `join`, `distinct` / `dropDuplicates`, `orderBy` / `sort`,
> `repartition`. Window functions with `partitionBy` too.
>
> **No shuffle (narrow):** `select`, `filter`, `withColumn`, `drop`, `union`, `coalesce`.
>
> I do not memorise the lists, I ask one question: **does an output partition need data from more
> than one input partition?** Yes means wide, which means a shuffle.
>
> Two traps: `describe()` **does** shuffle, because it aggregates. And `printSchema()` / `.columns`
> are not narrow operations at all, they are metadata and create no job.

---

**4. Why did `count()` cost two stages when `show(5)` cost one?**  `[ ]`

> `count()` produces one number that depends on every partition, so partial results must be
> combined: shuffle, two stages. `show(5)` is satisfied by **any** five rows, so Spark read one
> partition and stopped. Nothing combines, one stage.
>
> The numbers: `show(5)` read 16 MiB / 4096 rows. `count()` read 153 MB / 9,554,778 rows. Same
> DataFrame, same code above it, different action.

---

**5. Why does `collect()` on a raw DataFrame kill the driver?**  `[ ]`

> `collect()` moves **every row from every executor into the driver's JVM heap** as objects. The
> driver is one machine with fixed memory, 4 GB here. Cluster size is irrelevant: the whole result
> has to fit in that one heap.
>
> When it does not, the heap fills, GC runs, nothing can be freed because every row is still
> referenced, so the JVM burns CPU collecting forever instead of failing fast. I did this with
> `show(5_000_000)`: 4.68 GB resident, 52% CPU, 16 minutes, zero progress. `toPandas()` carries the
> same risk.
>
> The safe pattern is **aggregate first, collect second.** My `groupBy` reduced 9.5M rows to 262
> before I collected, and that was fine.


In [ ]:
# Release the session when you are finished. The UI on :4040 goes with it,
# so do this last.